In [1]:
import pandas as pd
import numpy as np
import glob
import os

In [2]:
# Locate all Parquet files

files = sorted(glob.glob("../data/cicids2017/*.parquet"))

print("Files found:", len(files))

# Create an empty list
frames = []

# Read every Parquet file
for f in files:

    print("Loading:", os.path.basename(f))

    d = pd.read_parquet(f)

    # Remove unnecessary spaces from column names
    d.columns = d.columns.str.strip()

    # Convert numerical columns to float32
    num = d.select_dtypes(include="number").columns
    d[num] = d[num].astype("float32")

    # Add dataset to list
    frames.append(d)

# Combine all datasets
df = pd.concat(frames, ignore_index=True)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

Files found: 8
Loading: Benign-Monday-no-metadata.parquet
Loading: Botnet-Friday-no-metadata.parquet
Loading: Bruteforce-Tuesday-no-metadata.parquet
Loading: DDoS-Friday-no-metadata.parquet
Loading: DoS-Wednesday-no-metadata.parquet
Loading: Infiltration-Thursday-no-metadata.parquet
Loading: Portscan-Friday-no-metadata.parquet
Loading: WebAttacks-Thursday-no-metadata.parquet

Dataset loaded successfully!
Dataset shape: (2313810, 78)


In [3]:
# Replace infinite values with missing values

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Remove rows containing missing values

df.dropna(inplace=True)

# Remove duplicate rows

df.drop_duplicates(inplace=True)

# Display final dataset dimensions

print(df.shape)

C:\Users\huawe\AppData\Local\Temp\ipykernel_14996\555578344.py:7: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.dropna(inplace=True)
C:\Users\huawe\AppData\Local\Temp\ipykernel_14996\555578344.py:11: Pandas4Warning: The inplace keyword in DataFrame.drop_duplicates is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop_duplicates(inplace=True)


(2199002, 78)


In [4]:
print("Dataset shape:", df.shape)

print("\nDataset columns:")
print(df.columns.tolist())

print("\nLabel column exists:", "Label" in df.columns)

Dataset shape: (2199002, 78)

Dataset columns:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 

In [5]:

# Display all attack categories and their record counts

print(df["Label"].value_counts())


Label
Benign                        1862525
DoS Hulk                       172846
DDoS                           128014
DoS GoldenEye                   10282
FTP-Patator                      5931
DoS slowloris                    5374
DoS Slowhttptest                 5228
SSH-Patator                      3219
PortScan                         1956
Web Attack � Brute Force         1470
Bot                              1437
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [6]:

# Step 1: Create a function to group attack labels

def group(label):

    # Normalize label to handle capitalization and spaces
    l = str(label).strip().lower()

    if l == "benign":
        return "BENIGN"

    if "web attack" in l:
        return "WebAttack"

    if l in ("ftp-patator", "ssh-patator"):
        return "BruteForce"

    if l.startswith("dos"):
        return "DoS"

    if l == "ddos":
        return "DDoS"

    if l == "portscan":
        return "PortScan"

    if l == "bot":
        return "Bot"

    return "Other"


In [7]:

# Step 2: Apply grouping function to every row

df["Group"] = df["Label"].apply(group)

# Display the new attack categories

print(df["Group"].value_counts())


Group
BENIGN        1862525
DoS            193730
DDoS           128014
BruteForce       9150
WebAttack        2143
PortScan         1956
Bot              1437
Other              47
Name: count, dtype: int64


In [8]:

# Step 3: Remove Other category

df = df[df["Group"] != "Other"].copy()

print("Dataset shape after removing Other:")
print(df.shape)

print("\nRemaining attack categories:")
print(df["Group"].value_counts())


Dataset shape after removing Other:
(2198955, 79)

Remaining attack categories:
Group
BENIGN        1862525
DoS            193730
DDoS           128014
BruteForce       9150
WebAttack        2143
PortScan         1956
Bot              1437
Name: count, dtype: int64


In [9]:
# Step 4: Remove original Label column

df = df.drop(columns=["Label"])

print("Remaining columns:")
print(df.columns.tolist())

print("\nDataset shape:")
print(df.shape)

Remaining columns:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag 

In [10]:

# Step 5: Find constant columns

const = [
    c for c in df.columns
    if c != "Group" and df[c].nunique() <= 1
]

print("Constant columns found:")
print(const)


Constant columns found:
['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']


In [11]:
# Step 6: Remove constant columns

df = df.drop(columns=const)

print("Dataset shape after removing constant columns:")
print(df.shape)

Dataset shape after removing constant columns:
(2198955, 70)


In [12]:
# Step 7: Save cleaned dataset

df.to_parquet("../data/clean.parquet")

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [13]:
# Verify saved dataset

print("Dataset shape:", df.shape)

print("\nConstant columns remaining:")
print([
    c for c in df.columns
    if c != "Group" and df[c].nunique() <= 1
])

print("\nAttack categories:")
print(df["Group"].value_counts())

print("\nSaved file exists:", os.path.exists("../data/clean.parquet"))

Dataset shape: (2198955, 70)

Constant columns remaining:
[]

Attack categories:
Group
BENIGN        1862525
DoS            193730
DDoS           128014
BruteForce       9150
WebAttack        2143
PortScan         1956
Bot              1437
Name: count, dtype: int64

Saved file exists: True
